In [0]:
from pyspark.sql import functions as F
silver = spark.table("workspace.vax_pipeline.silver_vax_malaysia")



In [0]:
# ---- Date dimension ----
dim_date = (silver.select("date").distinct()
    .withColumn("date_key",     F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("full_date",    F.col("date"))
    .withColumn("year",         F.year("date"))
    .withColumn("month",        F.month("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_name",     F.date_format("date", "EEEE")))

# ---- Fact table ----
fact = (silver
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .selectExpr(
        "date_key",
        "daily_partial", "daily_full", "daily_booster",
        "daily as daily_total",
        "cumul_partial", "cumul_full", "cumul as cumul_total"))


gold_summary = silver.select("date", "daily", "cumul")

write_opts = {"overwriteSchema": "true"}

(dim_date.write.format("delta").mode("overwrite")
    .options(**write_opts)
    .saveAsTable("workspace.vax_pipeline.dim_date"))

(fact.write.format("delta").mode("overwrite")
    .options(**write_opts)
    .saveAsTable("workspace.vax_pipeline.fact_vaccination"))

(gold_summary.write.format("delta").mode("overwrite")
    .options(**write_opts)
    .saveAsTable("workspace.vax_pipeline.gold_vax_malaysia"))

display(spark.table("workspace.vax_pipeline.fact_vaccination"))

date_key,daily_partial,daily_full,daily_booster,daily_total,cumul_partial,cumul_full,cumul_total
20210317,20799,67,0,20866,373588,269,373852
20210322,9721,14578,0,24299,428778,31449,460222
20210328,3684,12942,0,16626,459134,134735,593864
20210402,7574,22910,0,30484,511967,268540,780502
20210403,2670,14751,0,17421,514637,283291,797923
20210423,12222,6701,0,18923,791944,502960,1294896
20210503,24545,26751,0,51296,939136,598734,1537862
20210511,30555,18465,0,49020,1182881,734048,1916918
20210611,96427,30316,0,126743,2969700,1315410,4285098
20210616,206417,8310,0,214727,3699966,1497239,5197193


In [0]:
print("catalog:", spark.catalog.currentCatalog())
print("schema:", spark.catalog.currentDatabase())
print(spark.sql("SHOW TABLES IN workspace.vax_pipeline").show())

catalog: workspace
schema: default
+------------+-------------------+-----------+
|    database|          tableName|isTemporary|
+------------+-------------------+-----------+
|vax_pipeline|bronze_vax_malaysia|      false|
|vax_pipeline|           dim_date|      false|
|vax_pipeline|   fact_vaccination|      false|
|vax_pipeline|  gold_vax_malaysia|      false|
|vax_pipeline|silver_vax_malaysia|      false|
+------------+-------------------+-----------+

None
